In [1]:
pip install pandas numpy yfinance pandas_datareader matplotlib

In [2]:
import pandas as pd
import yfinance as yf
import pandas_datareader.data as pdr
import matplotlib.pyplot as plt
import numpy as np

# --- 1. 指標定義 ---
# (已註解掉 FRED 的 WTI_Oil 來修復 ValueError)
INDICATOR_MAP = {
    # 殖利率與利差
    'Yield_10Y': {'code': 'DGS10', 'source': 'fred', 'freq': 'D', 'trans': 'level'},
    'Yield_2Y': {'code': 'DGS2',  'source': 'fred', 'freq': 'D', 'trans': 'level'},

    # 通膨相關
    'Core_PCE_YoY': {'code': 'PCEPILFE', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},
    'CPI_YoY': {'code': 'CPIAUCSL', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},
    'PPI_YoY': {'code': 'PPIACO', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},
    'Core_CPI_YoY': {'code': 'CPILFESL', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},
    'PCE_YoY': {'code': 'PCEPI', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},
    'Inflation_Exp_10Y': {'code': 'T10YIE', 'source': 'fred', 'freq': 'D', 'trans': 'level'},
    # 'WTI_Oil': {'code': 'DCOILWTICO', 'source': 'fred', 'freq': 'D', 'trans': 'level'}, # <-- 已註解

    # 勞動力市場
    'Unemployment_Rate': {'code': 'UNRATE', 'source': 'fred', 'freq': 'M', 'trans': 'level'},
    'Avg_Hourly_Earnings_YoY': {'code': 'CES0500000003', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},
    'Initial_Claims': {'code': 'ICSA', 'source': 'fred', 'freq': 'W', 'trans': 'level'},
    'Non_Farm_Payrolls': {'code': 'PAYEMS', 'source': 'fred', 'freq': 'M', 'trans': 'level'},
    'JOLTS_Openings': {'code': 'JTSJOL', 'source': 'fred', 'freq': 'M', 'trans': 'level'},
    'Labor_Participation': {'code': 'CIVPART', 'source': 'fred', 'freq': 'M', 'trans': 'level'},

    # 經濟活動
    'ISM_Manufacturing': {'code': 'INDPRO', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'}, # 替換為: 工業生產指數 (年增率)
    'ISM_Services': {'code': 'PCESV', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},    # T替換為: 個人服務消費 (年增率)
    'Real_GDP_YoY': {'code': 'GDPC1', 'source': 'fred', 'freq': 'Q', 'trans': 'yoy'},
    'Michigan_Sentiment': {'code': 'UMCSENT', 'source': 'fred', 'freq': 'M', 'trans': 'level'},
    'Retail_Sales_YoY': {'code': 'RSAFS', 'source': 'fred', 'freq': 'M', 'trans': 'yoy'},

    # 金融與市場
    'Fed_Total_Assets': {'code': 'WALCL', 'source': 'fred', 'freq': 'W', 'trans': 'level'},
    'Financial_Conditions_Idx': {'code': 'NFCI', 'source': 'fred', 'freq': 'W', 'trans': 'level'},
    'Dollar_Index': {'code': 'DTWEXBGS', 'source': 'fred', 'freq': 'D', 'trans': 'level'},
    'Copper_Price': {'code': 'HG=F', 'source': 'yfinance', 'freq': 'D', 'trans': 'level'},
    'WTI_Oil_yf': {'code': 'CL=F', 'source': 'yfinance', 'freq': 'D', 'trans': 'level'},

    # 債務
    'Debt_to_GDP': {'code': 'GFDEGDQ188S', 'source': 'fred', 'freq': 'Q', 'trans': 'level'},
}


# === 輔助函數 ===
def get_sampling_days(freq):
    return {'M': 365, 'Q': 365, 'W': 364, 'D': 365}.get(freq, 365)


# === 資料處理 ===
def fetch_all_data(indicators, start_date, end_date):
    print("正在下載所有數據...")
    fred_codes = [v['code'] for v in indicators.values() if v['source'] == 'fred']
    yf_codes = [v['code'] for v in indicators.values() if v['source'] == 'yfinance']

    df_fred = pdr.get_data_fred(fred_codes, start=start_date, end=end_date)

    if len(yf_codes) > 0:
        # 修正 FutureWarning
        df_yf = yf.download(yf_codes, start=start_date, end=end_date, auto_adjust=True)['Close']
        if isinstance(df_yf, pd.Series):
            df_yf = df_yf.to_frame()
    else:
        df_yf = pd.DataFrame()

    df_yf = df_yf.rename(columns={v['code']: k for k, v in indicators.items() if v['source'] == 'yfinance'})
    df_fred = df_fred.rename(columns={v['code']: k for k, v in indicators.items() if v['source'] == 'fred'})

    df_combined = pd.concat([df_fred, df_yf], axis=1).asfreq('D')
    print("數據下載完成。")
    return df_combined


def preprocess_and_transform(df, indicators):
    print("正在進行數據預處理 (YoY 計算、頻率統一與前值填充)...")
    df_transformed = pd.DataFrame(index=pd.date_range(df.index.min(), df.index.max(), freq='D'))

    for name, cfg in indicators.items():
        if name not in df.columns:
            continue
        s = df[name].dropna().copy()

        # resample to daily
        if cfg['freq'] != 'D':
            s = s.resample('D').ffill()

        # 計算 YoY
        if cfg['trans'] == 'yoy':
            s_yoy = (s / s.shift(get_sampling_days(cfg['freq'])) - 1) * 100
            df_transformed[name] = s_yoy
        else:
            df_transformed[name] = s

    # === 衍生指標 ===
    # 銅油比 (依賴 'WTI_Oil_yf')
    if 'Copper_Price' in df_transformed and 'WTI_Oil_yf' in df_transformed:
        df_transformed['Copper_Price'] = df_transformed['Copper_Price'].ffill()
        df_transformed['WTI_Oil_yf'] = df_transformed['WTI_Oil_yf'].ffill()
        df_transformed['Copper_Oil_Ratio'] = df_transformed['Copper_Price'] / df_transformed['WTI_Oil_yf']
        # 這裡將 'WTI_Oil_yf' 重新命名為 'WTI_Oil'，這就是為什麼我們必須刪除 FRED 的 WTI_Oil
        df_transformed = df_transformed.rename(columns={'WTI_Oil_yf': 'WTI_Oil'})

    # 10Y–2Y 利差 (依賴 'Yield_10Y')
    if 'Yield_10Y' in df_transformed and 'Yield_2Y' in df_transformed:
        df_transformed['Yield_Spread_10Y_2Y'] = df_transformed['Yield_10Y'] - df_transformed['Yield_2Y']

    # 最終填補
    df_final = df_transformed.asfreq('D').ffill().dropna(how='all')
    print("數據預處理完成。")
    return df_final


# === 相關性分析 ===
def calculate_positive_lag_correlation(df, base_col, target_col, max_lag_periods):
    """
    注意：max_lag_periods 現在代表 '月' (如果傳入的是月資料)。
    """
    results = []
    data = df[[base_col, target_col]].dropna().copy()
    for lag in range(0, max_lag_periods + 1):
        temp = data.copy()
        temp[f'{target_col}_lagged'] = data[target_col].shift(lag)
        temp = temp.dropna()
        corr = temp[base_col].corr(temp[f'{target_col}_lagged'])
        results.append({'lag': lag, 'correlation': corr})
    return pd.DataFrame(results).set_index('lag')


def analyze_and_plot(corr_df, base_col_name, target_col_name, plot=False):
    """
    修改了輸出的文字，從 '天' 改為 '月'。
    """
    if corr_df.empty or corr_df['correlation'].isnull().all():
        print(f"\n--- 分析結果: {base_col_name} vs. {target_col_name} ---")
        print("數據不足，無法計算相關性。")
        return None

    best_lag_months = corr_df['correlation'].abs().idxmax()
    best_corr = corr_df.loc[best_lag_months, 'correlation']

    print(f"\n--- 分析結果: {base_col_name} vs. {target_col_name} ---")
    print(f"最大相關性 (絕對值) 發生在 Lag = {best_lag_months} 月，相關係數 = {best_corr:.4f}")

    if plot:
        plt.figure(figsize=(12, 6))
        corr_df['correlation'].plot()
        plt.axvline(x=best_lag_months, color='red', linestyle='--', label=f'Max |corr|={best_corr:.2f} @ lag={best_lag_months}')
        plt.xlabel("Lag (Months)") # X軸標籤
        plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

    return {'indicator': target_col_name, 'best_lag_months': best_lag_months, 'correlation': best_corr}


# === 主程式 ===
if __name__ == "__main__":
    START_DATE = "2005-01-01"
    END_DATE = "2025-11-05" # 腳本會自動抓到昨天 (2025-11-07)
    MAX_LAG_MONTHS = 24  # 最大落後 24 個月
    BASE_COLUMN = 'Yield_2Y'

    # 1. 抓取並轉換為每日資料
    df_raw = fetch_all_data(INDICATOR_MAP, START_DATE, END_DATE)
    df_final_daily = preprocess_and_transform(df_raw, INDICATOR_MAP)

    # 2. 將每日資料降頻為月均值
    print("\n正在將每日數據降頻為月均值...")
    # 使用 'ME' (Month-End) 來取代 'M'
    df_monthly_final = df_final_daily.resample('ME').mean()
    df_monthly_final = df_monthly_final.dropna(how='all') # 移除全為 NaN 的月份
    print("降頻完成。")
    print(f"月均值資料維度: {df_monthly_final.shape}")

    # 3. 使用月均值資料進行相關性分析

    # 建立一個排除列表
    EXCLUDE_COLS = [
        BASE_COLUMN,            # 排除自己 (Yield_2Y)
        'Yield_10Y',            # 依您要求排除
        'Yield_Spread_10Y_2Y'   # 依您要求排除
    ]

    all_results = []
    for col in df_monthly_final.columns:
        # 檢查 col 是否在排除列表中
        if col in EXCLUDE_COLS:
            continue

        # 檢查數據是否全為空
        if df_monthly_final[col].isnull().all():
            print(f"指標 {col} 數據為空，跳過。")
            continue

        corr_df = calculate_positive_lag_correlation(df_monthly_final, BASE_COLUMN, col, MAX_LAG_MONTHS)
        res = analyze_and_plot(corr_df, BASE_COLUMN, col, plot=False)
        if res:
            all_results.append(res)

    # 4. 總結報告
    results_df = pd.DataFrame(all_results)
    if not results_df.empty:
        results_df['abs_correlation'] = results_df['correlation'].abs()
        results_df = results_df.sort_values('abs_correlation', ascending=False)
        print("\n\n===== 總結報告 (基於月均值) =====")
        print(results_df[['indicator', 'best_lag_months', 'correlation', 'abs_correlation']].to_string(index=False))
    else:
        print("\n\n===== 總結報告 (基於月均值) =====")
        print("未產生任何有效的相關性分析結果。")

正在下載所有數據...


[*********************100%***********************]  2 of 2 completed


數據下載完成。
正在進行數據預處理 (YoY 計算、頻率統一與前值填充)...
數據預處理完成。

正在將每日數據降頻為月均值...
降頻完成。
月均值資料維度: (251, 27)

--- 分析結果: Yield_2Y vs. Core_PCE_YoY ---
最大相關性 (絕對值) 發生在 Lag = 16 月，相關係數 = 0.8108

--- 分析結果: Yield_2Y vs. CPI_YoY ---
最大相關性 (絕對值) 發生在 Lag = 16 月，相關係數 = 0.6733

--- 分析結果: Yield_2Y vs. PPI_YoY ---
最大相關性 (絕對值) 發生在 Lag = 16 月，相關係數 = 0.3957

--- 分析結果: Yield_2Y vs. Core_CPI_YoY ---
最大相關性 (絕對值) 發生在 Lag = 17 月，相關係數 = 0.7866

--- 分析結果: Yield_2Y vs. PCE_YoY ---
最大相關性 (絕對值) 發生在 Lag = 15 月，相關係數 = 0.6990

--- 分析結果: Yield_2Y vs. Inflation_Exp_10Y ---
最大相關性 (絕對值) 發生在 Lag = 11 月，相關係數 = 0.4940

--- 分析結果: Yield_2Y vs. Unemployment_Rate ---
最大相關性 (絕對值) 發生在 Lag = 5 月，相關係數 = -0.6326

--- 分析結果: Yield_2Y vs. Avg_Hourly_Earnings_YoY ---
最大相關性 (絕對值) 發生在 Lag = 24 月，相關係數 = 0.7225

--- 分析結果: Yield_2Y vs. Initial_Claims ---
最大相關性 (絕對值) 發生在 Lag = 4 月，相關係數 = -0.2906

--- 分析結果: Yield_2Y vs. Non_Farm_Payrolls ---
最大相關性 (絕對值) 發生在 Lag = 8 月，相關係數 = 0.4567

--- 分析結果: Yield_2Y vs. JOLTS_Openings ---
最大相關性 (絕對值) 發生在 Lag = 24 月，相關係數 =